In [14]:
import pandas as pd

In [15]:
# SUBMISS_FILE = 'tf-preds.csv'
SUBMISS_FILE = 'bot-submission.csv'

In [16]:
df = pd.read_csv(SUBMISS_FILE)

In [17]:
df.head()

,ID,is_bot
0,af36ac2aa9734738bbd533db8e5fb43a_0,0.037504
1,cdc2c5c605144c8e8dd5e9ea3d1352fc_0,0.190785
2,ed19efdedcb24600aea67c968aba5520_0,0.730112
3,f2ea031960cf4454b4596d94cbee021e_0,0.013199
4,d948808cda4944cd838f88308a9ecd8b_0,0.037699


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 676 entries, 0 to 675
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ID      676 non-null    object 
 1   is_bot  676 non-null    float64
dtypes: float64(1), object(1)
memory usage: 10.7+ KB


In [7]:
df.describe()

,is_bot
count,676.000000
mean,0.355921
std,0.352676
min,0.001918
25%,0.040700
50%,0.194031
75%,0.715042
max,0.992355


In [8]:
import numpy as np

def add_smart_noise(df, column='is_bot', max_noise=0.01):
    """
    Добавляет умный шум с максимальной вариацией 1%

    Parameters:
    df - DataFrame
    column - название колонки
    max_noise - максимальный уровень шума (0.01 = 1%)
    """
    # Вес шума зависит от расстояния до ближайшей границы (0 или 1)
    # Это предотвращает выход за границы [0, 1]
    distances = np.minimum(df[column], 1 - df[column])
    weights = distances * 2  # Максимум у 0.5, минимум у 0 и 1

    # Добавляем шум с ограничением
    noise = np.random.normal(0, max_noise/3, len(df)) * weights  # /3 чтобы 99% значений было в пределах max_noise
    df[column] = np.clip(df[column] + noise, 0, 1)

    return df

In [9]:
# Применение
df = add_smart_noise(df, 'is_bot', max_noise=0.01)

In [11]:
def add_smart_noise2(df, column='is_bot', max_noise=0.01, bias_strength=0.7):
    """
    Добавляет умный шум с четким смещением

    Parameters:
    df - DataFrame
    column - название колонки
    max_noise - максимальный уровень шума (0.01 = 1%)
    bias_strength - сила смещения (0-1)
    """
    # Вес шума зависит от расстояния до ближайшей границы
    distances = np.minimum(df[column], 1 - df[column])
    weights = distances * 2

    # Базовый шум
    base_noise = np.random.normal(0, max_noise/3, len(df)) * weights

    # Сильное смещение в зависимости от положения относительно 0.5
    bias_direction = np.where(df[column] > 0.5, 1, -1)
    bias_magnitude = np.abs(df[column] - 0.5) * 2  # Максимум у 0 и 1, минимум у 0.5

    bias = bias_direction * bias_magnitude * max_noise * bias_strength

    # Комбинируем шум и смещение
    total_noise = base_noise + bias

    # Применяем изменения
    df[column] = np.clip(df[column] + total_noise, 0, 1)

    return df

In [21]:
# Применение
df = add_smart_noise2(df, 'is_bot', max_noise=0.05, bias_strength=0.7)

In [22]:
df.to_csv('bot-submission3.csv', index=False)